# Bag-of-Words Decision Tree Experiments

Runs several research-style Bag-of-Words + Decision Tree experiments, logs all runs to MLflow, and saves Kaggle submissions. The feature idea is based on separate premise/hypothesis vectors plus optional overlap, difference, and language one-hot features.

In [ ]:
import json
from pathlib import Path

import joblib
import mlflow
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

## Load Data

Prefer DVC-versioned processed splits. If this notebook is copied into Kaggle, fallback paths can be used.

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
KAGGLE_INPUT_DIR = Path('/kaggle/input/competitions/contradictory-my-dear-watson')

if (PROCESSED_DIR / 'train_split.csv').exists():
    train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
    sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')
    data_source = 'dvc_processed_split'
elif (KAGGLE_INPUT_DIR / 'train.csv').exists():
    full_train_df = pd.read_csv(KAGGLE_INPUT_DIR / 'train.csv')
    test_df = pd.read_csv(KAGGLE_INPUT_DIR / 'test.csv')
    sample_submission = pd.read_csv(KAGGLE_INPUT_DIR / 'sample_submission.csv')
    train_df, val_df = train_test_split(
        full_train_df,
        test_size=0.2,
        random_state=42,
        stratify=full_train_df['label'],
    )
    data_source = 'kaggle_input_fallback_split'
else:
    raise FileNotFoundError('Could not find DVC processed splits or Kaggle input files.')

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

len(train_df), len(val_df), len(test_df), data_source

## Feature Builder

The vocabulary is shared between premise and hypothesis, but their feature blocks stay separate. Extra blocks can be added per experiment.

In [ ]:
def normalize_text(value):
    return ' '.join(str(value).lower().split())


def fit_vectorizer(train_frame, ngram_range, max_features):
    vectorizer = CountVectorizer(
        lowercase=True,
        ngram_range=ngram_range,
        max_features=max_features,
    )
    vectorizer.fit(
        pd.concat([
            train_frame['premise'].fillna('').map(normalize_text),
            train_frame['hypothesis'].fillna('').map(normalize_text),
        ])
    )
    return vectorizer


def make_features(frame, vectorizer, language_columns, use_overlap, use_difference, use_language):
    premise_vec = vectorizer.transform(frame['premise'].fillna('').map(normalize_text))
    hypothesis_vec = vectorizer.transform(frame['hypothesis'].fillna('').map(normalize_text))

    blocks = [premise_vec, hypothesis_vec]

    if use_overlap:
        blocks.append(premise_vec.multiply(hypothesis_vec))

    if use_difference:
        blocks.append(abs(premise_vec - hypothesis_vec))

    if use_language:
        lang_vec = csr_matrix(
            pd.get_dummies(frame['lang_abv'])
            .reindex(columns=language_columns, fill_value=0)
            .astype('int8')
        )
        blocks.append(lang_vec)

    return hstack(blocks, format='csr')

## Experiment Grid

These runs are intentionally simple. The point is to populate MLflow with comparable baseline runs.

In [ ]:
EXPERIMENTS = [
    {
        'run_name': 'bow_tree_unigram_depth20_pair_only',
        'ngram_range': (1, 1),
        'max_features': 20_000,
        'max_depth': 20,
        'use_overlap': False,
        'use_difference': False,
        'use_language': False,
    },
    {
        'run_name': 'bow_tree_bigram_depth30_pair_only',
        'ngram_range': (1, 2),
        'max_features': 30_000,
        'max_depth': 30,
        'use_overlap': False,
        'use_difference': False,
        'use_language': False,
    },
    {
        'run_name': 'bow_tree_bigram_depth50_overlap_difference',
        'ngram_range': (1, 2),
        'max_features': 50_000,
        'max_depth': 50,
        'use_overlap': True,
        'use_difference': True,
        'use_language': False,
    },
    {
        'run_name': 'bow_tree_bigram_depth50_overlap_difference_language',
        'ngram_range': (1, 2),
        'max_features': 50_000,
        'max_depth': 50,
        'use_overlap': True,
        'use_difference': True,
        'use_language': True,
    },
]

language_columns = sorted(train_df['lang_abv'].dropna().unique().tolist())
results = []

## Run Experiments

In [ ]:
for params in EXPERIMENTS:
    run_name = params['run_name']
    print(f'Running {run_name}...')

    vectorizer = fit_vectorizer(
        train_df,
        ngram_range=params['ngram_range'],
        max_features=params['max_features'],
    )

    x_train = make_features(
        train_df,
        vectorizer,
        language_columns,
        params['use_overlap'],
        params['use_difference'],
        params['use_language'],
    )
    x_val = make_features(
        val_df,
        vectorizer,
        language_columns,
        params['use_overlap'],
        params['use_difference'],
        params['use_language'],
    )
    x_test = make_features(
        test_df,
        vectorizer,
        language_columns,
        params['use_overlap'],
        params['use_difference'],
        params['use_language'],
    )

    model = DecisionTreeClassifier(
        random_state=42,
        max_depth=params['max_depth'],
    )
    model.fit(x_train, train_df['label'].astype(int))

    val_preds = model.predict(x_val)
    metrics = compute_all_metrics(val_df, val_preds)
    test_preds = model.predict(x_test)

    submission = build_submission(sample_submission, test_preds)
    submission_path = Path('../submissions') / f'{run_name}.csv'
    save_submission(submission, submission_path)

    model_path = Path('../submissions') / f'{run_name}_model.joblib'
    vectorizer_path = Path('../submissions') / f'{run_name}_vectorizer.joblib'
    language_columns_path = Path('../submissions') / f'{run_name}_language_columns.json'

    joblib.dump(model, model_path)
    joblib.dump(vectorizer, vectorizer_path)
    language_columns_path.write_text(json.dumps(language_columns, indent=2), encoding='utf-8')

    serving_type = (
        'sklearn_text_pair_vectorizer'
        if not params['use_overlap'] and not params['use_difference'] and not params['use_language']
        else 'sklearn_engineered_text_pair_features'
    )
    metadata = {
        'run_name': run_name,
        'data_source': data_source,
        'serving_type': serving_type,
        'label_mapping': {0: 'entailment', 1: 'neutral', 2: 'contradiction'},
        'preprocessing': 'lowercase_and_normalize_whitespace',
        'feature_order': ['premise_vec', 'hypothesis_vec']
        + (['overlap_vec'] if params['use_overlap'] else [])
        + (['difference_vec'] if params['use_difference'] else [])
        + (['language_one_hot'] if params['use_language'] else []),
        'language_columns': language_columns,
        'params': {**params, 'ngram_range': list(params['ngram_range'])},
    }

    mlflow_params = {
        **params,
        'ngram_range': str(params['ngram_range']),
        'model': 'DecisionTreeClassifier',
        'random_state': 42,
        'data_source': data_source,
        'serving_type': serving_type,
    }

    with start_notebook_run(
        run_name,
        tags={
            'model_family': 'decision_tree',
            'features': 'bow_pair_overlap_difference_language',
            'serving_type': serving_type,
        },
    ):
        mlflow.log_params(mlflow_params)
        log_metrics(metrics)
        log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
        mlflow.log_artifact(str(submission_path), artifact_path='submissions')
        log_model_artifacts(
            artifacts={
                'model.joblib': model_path,
                'vectorizer.joblib': vectorizer_path,
                'language_columns.json': language_columns_path,
            },
            metadata=metadata,
            artifact_path='model',
        )

    results.append({
        'run_name': run_name,
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'precision_macro': metrics['precision_macro'],
        'recall_macro': metrics['recall_macro'],
        'submission_path': str(submission_path),
        'serving_type': serving_type,
    })

pd.DataFrame(results).sort_values('f1_macro', ascending=False)